In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler, LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
df = pd.read_csv('housing.csv')
df.head(2)

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df = df.dropna()
df.shape,df.isnull().sum()

In [ ]:
df.describe()

In [ ]:
df.ocean_proximity.value_counts()

In [ ]:
label = LabelEncoder()
df.ocean_proximity = label.fit_transform(df.ocean_proximity)
df.head()

In [ ]:
df.info()

In [ ]:
for i in df.columns:
    sns.histplot(data=df, x=i,kde=True, bins=20)
    plt.xticks(rotation=45)
    plt.show()

In [ ]:
for i in df.columns:
    sns.boxplot(data=df, x=i)
    plt.xticks(rotation=45)
    plt.show()

In [ ]:
sns.heatmap(df.corr(), annot=True)
plt.show()

In [ ]:
y = df.median_house_value
x = df.drop('median_house_value', axis=1)

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [ ]:
min_max = MinMaxScaler()
x_train = min_max.fit_transform(x_train)
x_test = min_max.transform(x_test)

In [ ]:
class PriceModel(nn.Module):
    def __init__(self,in_par):
        super(PriceModel,self).__init__()
        self.flat = nn.Flatten()
        self.seq = nn.Sequential(
            nn.Linear(in_par, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64,32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32,16),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(16,1)      
            )
    
    def forward(self, x):
        x1 = self.flat(x)
        return self.seq(x1)

In [ ]:
x_train_tensor = torch.tensor(data=x_train, dtype=torch.float32)
x_test_tensor = torch.tensor(data=x_test, dtype=torch.float32)
y_train_tensor = torch.tensor(data=y_train.values, dtype=torch.float32)
y_test_tensor = torch.tensor(data=y_test.values, dtype=torch.float32)

In [ ]:
train_df = TensorDataset(x_train_tensor, y_train_tensor)
test_df = TensorDataset(x_test_tensor, y_test_tensor)

train_loader = DataLoader(train_df,batch_size=32, shuffle=True)
test_loader = DataLoader(test_df, batch_size=32, shuffle=False)

In [ ]:
x_train.shape

In [ ]:
model = PriceModel(in_par=9)
model

In [ ]:
criteria = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr = 0.001)

In [ ]:
epochs = 100

for ep in range(epochs):
    
    model.train()
    run_loss = 0.0

    for ip, target in train_loader:
        optimizer.zero_grad()
        op = model(ip)
        loss = criteria(op, target)
        loss.backward()
        optimizer.step()
        run_loss += loss.item()*ip.size(0)
    
    ep_loss = run_loss/len(train_loader.dataset)

    if(ep+1)%10 == 0:
        print(f"Epoch [{ep+1}/{epochs}], Loss: {ep_loss:.4f}")

In [ ]:
model.eval()
with torch.no_grad():
    preds = model(x_train_tensor)

    # loss (correct)
    test_loss = criteria(preds, y_train_tensor.view(-1))
    print(f"\n✅ Train Loss: {test_loss.item():.4f}")

    # convert logits → predicted class
    predicted_classes = torch.argmax(preds, dim=1)

    # convert to numpy
    y_true = y_train_tensor.view(-1).cpu().numpy()
    y_pred = predicted_classes.cpu().numpy()

In [ ]:
train_losses = []
val_losses = []

for epoch in range(epochs):
    # ---- Training ----
    model.train()
    optimizer.zero_grad()
    
    outputs = model(x_train_tensor)
    loss = criteria(outputs, y_train_tensor)
    
    loss.backward()
    optimizer.step()
    
    train_losses.append(loss.item())

    # ---- Validation ----
    model.eval()
    with torch.no_grad():
        val_outputs = model(x_test_tensor)
        val_loss = criteria(val_outputs, y_test_tensor.view(-1))
        val_losses.append(val_loss.item())
import matplotlib.pyplot as plt

plt.plot(train_losses, label='Training Loss')
plt.plot(val_losses, label='Validation Loss')

plt.title('Training vs Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.show()